In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.2

In [2]:
from pathlib import Path

datasets = {
    "Roboflow": "/kaggle/input/datasets/sampreetvaidya/warehouse-box-count/warehouse-box-count-dataset",
    "My": "/kaggle/input/datasets/tuyennguyen21pt/dataset/dataset",
    "Manual": "/kaggle/input/datasets/tuyennguyen21pt/dataset-manual/dataset",
}

img_ext = {".jpg", ".jpeg", ".png", ".webp"}

for name, path in datasets.items():
    path = Path(path)

    images = sum(1 for f in path.rglob("*") if f.suffix.lower() in img_ext)
    labels = sum(1 for f in path.rglob("*.txt"))

    print(f"{name}:")
    print(f"  Images: {images}")
    print(f"  Labels: {labels}")
    print()

Roboflow:
  Images: 340
  Labels: 342

My:
  Images: 1049
  Labels: 1049

Manual:
  Images: 21
  Labels: 23



In [3]:
from pathlib import Path

img_exts = {".jpg", ".jpeg", ".png", ".webp"}

dataset = Path("/kaggle/working/dataset")   # đổi thành thư mục cần xử lý

images = {f.stem: f for f in dataset.rglob("*") if f.suffix.lower() in img_exts}
labels = {f.stem: f for f in dataset.rglob("*.txt")}

print("Images:", len(images))
print("Labels:", len(labels))

for stem in images.keys() - labels.keys():
    images[stem].unlink()
    print("Removed image:", images[stem])

for stem in labels.keys() - images.keys():
    labels[stem].unlink()
    print("Removed label:", labels[stem])

print("Done!")

Images: 0
Labels: 0
Done!


In [4]:
import kagglehub

path = kagglehub.dataset_download("sampreetvaidya/warehouse-box-count")

print(path)

/kaggle/input/datasets/sampreetvaidya/warehouse-box-count


In [5]:
import os

dataset_path = "/kaggle/input/datasets/sampreetvaidya/warehouse-box-count"

for root, dirs, files in os.walk(dataset_path):
    print(root)
    print(files[:5])

/kaggle/input/datasets/sampreetvaidya/warehouse-box-count
[]
/kaggle/input/datasets/sampreetvaidya/warehouse-box-count/warehouse-box-count-dataset
['README.dataset.txt', 'README.roboflow.txt']
/kaggle/input/datasets/sampreetvaidya/warehouse-box-count/warehouse-box-count-dataset/valid
['IMG_1808_JPG.rf.6e4d0001297912790c607bfa997bb182.txt', 'IMG_1672_JPG.rf.f7b441bd6dc9d303eceb6e408eea1a89.jpg', 'IMG_1697_JPG.rf.7ddeb7cb7008a9e9e3676888f1220acf.txt', 'IMG_1786_JPG.rf.973e514282833f552382062fb5e83365.txt', 'IMG_1659_JPG.rf.8828f83c24d93c65a3fa50405106f2c0.txt']
/kaggle/input/datasets/sampreetvaidya/warehouse-box-count/warehouse-box-count-dataset/test
['IMG_1682_JPG.rf.13680c043fe5ca84f758db85c0b202c1.jpg', 'IMG_1665_JPG.rf.706ab7f5ca2f9824475ede390534d002.txt', 'IMG_1791_JPG.rf.83170dfac376ab861d1cc8e4e3a00db4.jpg', 'IMG_1809_JPG.rf.b453585c710773b072f29b551bb02ed6.jpg', '_darknet.labels']
/kaggle/input/datasets/sampreetvaidya/warehouse-box-count/warehouse-box-count-dataset/train
['conve

In [6]:
from ultralytics import YOLO
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
import shutil

src = "/kaggle/input/datasets/tuyennguyen21pt/dataset"
dst = "/kaggle/working/dataset"

shutil.copytree(src, dst, dirs_exist_ok=True)

print("Copied!")

Copied!


In [8]:
import os

dataset_path = "/kaggle/working/dataset/dataset"

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if not file.endswith(".txt"):
            continue

        txt_path = os.path.join(root, file)

        new_lines = []

        with open(txt_path, "r") as f:
            for line in f:
                parts = list(map(float, line.strip().split()))

                # Polygon: class x1 y1 x2 y2 x3 y3 x4 y4
                if len(parts) == 9:
                    cls = int(parts[0])
                    xs = parts[1::2]
                    ys = parts[2::2]

                    xmin = min(xs)
                    xmax = max(xs)
                    ymin = min(ys)
                    ymax = max(ys)

                    xc = (xmin + xmax) / 2
                    yc = (ymin + ymax) / 2
                    w = xmax - xmin
                    h = ymax - ymin

                    new_lines.append(
                        f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
                    )

                # Đã là YOLO thì giữ nguyên
                elif len(parts) == 5:
                    new_lines.append(line.strip())

        # Ghi đè file
        with open(txt_path, "w") as f:
            f.write("\n".join(new_lines))

print("Hoàn thành chuyển đổi!")

Hoàn thành chuyển đổi!


In [9]:
import os
import shutil
from pathlib import Path

# Dataset Roboflow
rf = "/kaggle/input/datasets/sampreetvaidya/warehouse-box-count/warehouse-box-count-dataset"

# Dataset của bạn
my = "/kaggle/working/dataset/dataset"

# Dataset manual
manual = "/kaggle/input/datasets/tuyennguyen21pt/dataset-manual/dataset"

# Dataset mới
out = "/kaggle/working/combined_dataset"

for split in ["train", "valid"]:
    os.makedirs(f"{out}/{split}/images", exist_ok=True)
    os.makedirs(f"{out}/{split}/labels", exist_ok=True)


def copy_yolo(src_split, dst_split):
    src = Path(src_split)

    for file in src.iterdir():
        if file.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:

            shutil.copy(
                file,
                f"{dst_split}/images/{file.name}"
            )

            label = file.with_suffix(".txt")

            if label.exists():
                shutil.copy(
                    label,
                    f"{dst_split}/labels/{label.name}"
                )


# Dataset chuẩn gồm images/labels
def copy_standard(src, dst):
    shutil.copytree(
        f"{src}/images",
        f"{dst}/images",
        dirs_exist_ok=True
    )

    shutil.copytree(
        f"{src}/labels",
        f"{dst}/labels",
        dirs_exist_ok=True
    )


# =========================
# Copy Roboflow
# =========================
copy_yolo(f"{rf}/train", f"{out}/train")
copy_yolo(f"{rf}/valid", f"{out}/valid")

# =========================
# Copy dataset của bạn
# =========================
copy_standard(f"{my}/train", f"{out}/train")
copy_standard(f"{my}/valid", f"{out}/valid")

# =========================
# Copy dataset manual -> train
# =========================
copy_yolo(manual, f"{out}/train")

In [10]:
from glob import glob

print(len(glob("/kaggle/working/combined_dataset/train/images/*")))
print(len(glob("/kaggle/working/combined_dataset/train/labels/*")))

print(len(glob("/kaggle/working/combined_dataset/valid/images/*")))
print(len(glob("/kaggle/working/combined_dataset/valid/labels/*")))

1158
1158
238
238


In [11]:
import yaml

data = {
    "train": "/kaggle/working/combined_dataset/train/images",
    "val": "/kaggle/working/combined_dataset/valid/images",
    "nc": 1,
    "names": ["box"]
}

with open("/kaggle/working/combined_dataset/data.yaml", "w") as f:
    yaml.dump(data, f, sort_keys=False)

print("Created data.yaml")

Created data.yaml


In [12]:
model = YOLO("yolo11m.pt")

In [ ]:
from pathlib import Path

# Chuẩn hoá TOÀN BỘ nhãn của dataset gộp (train + valid), từ mọi nguồn:
#  - Nếu là polygon/segmentation (class + n cặp toạ độ, > 5 token) -> đổi sang bbox.
#  - Nếu đã là bbox (5 token) -> giữ nguyên toạ độ.
#  - Luôn ép class về 0 để khớp data.yaml (nc:1, names:['box']).
# Chạy trên combined_dataset nên áp dụng đồng nhất cho Roboflow + My + Manual,
# tránh sót polygon nhiều đỉnh hay class index lạ (gốc rễ lỗi model 2 lớp '-'/'box').
label_dirs = [
    "/kaggle/working/combined_dataset/train/labels",
    "/kaggle/working/combined_dataset/valid/labels",
]

converted = 0
kept = 0

for label_dir in label_dirs:
    for txt_file in Path(label_dir).glob("*.txt"):
        new_lines = []

        with open(txt_file, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                parts = line.split()

                # Đã là bbox YOLO: class xc yc w h -> chỉ ép class = 0.
                if len(parts) == 5:
                    new_lines.append("0 " + " ".join(parts[1:]))
                    kept += 1

                # Polygon: class + các cặp (x, y) -> bao hình chữ nhật nhỏ nhất.
                elif len(parts) > 5 and (len(parts) - 1) % 2 == 0:
                    coords = list(map(float, parts[1:]))
                    xs = coords[0::2]
                    ys = coords[1::2]
                    xmin, xmax = min(xs), max(xs)
                    ymin, ymax = min(ys), max(ys)
                    xc = (xmin + xmax) / 2
                    yc = (ymin + ymax) / 2
                    w = xmax - xmin
                    h = ymax - ymin
                    new_lines.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
                    converted += 1

                # Định dạng lạ -> bỏ qua (log để biết).
                else:
                    print("Bỏ dòng không hợp lệ:", txt_file.name, "->", line)

        with open(txt_file, "w") as f:
            f.write("\n".join(new_lines) + ("\n" if new_lines else ""))

print(f"Chuẩn hoá xong. bbox giữ nguyên: {kept} | polygon -> bbox: {converted}")
print("Toàn bộ class đã ép về 0 (khớp names:['box']).")

In [14]:
from pathlib import Path

ok = True

for txt in Path("/kaggle/working/combined_dataset/train/labels").glob("*.txt"):
    with open(txt) as f:
        for line in f:
            if len(line.split()) != 5:
                print("Lỗi:", txt)
                ok = False
                break

print("Done:", ok)

Done: True


In [15]:
from pathlib import Path

seg = 0
box = 0

for txt in Path("/kaggle/working/combined_dataset/train/labels").glob("*.txt"):
    with open(txt) as f:
        for line in f:
            n = len(line.split())
            if n > 5:
                seg += 1
            elif n == 5:
                box += 1

print("Boxes:", box)
print("Segments:", seg)

Boxes: 17807
Segments: 0


In [16]:
model.train(
    data="/kaggle/working/combined_dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16
)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/combined_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b02181726f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [17]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2079.8±1422.4 MB/s, size: 302.2 KB)
val: Scanning /kaggle/working/combined_dataset/valid/labels.cache... 238 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 238/238 99.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.6it/s 9.5s
                   all        238       1797      0.848      0.734      0.818       0.64
Speed: 1.4ms preprocess, 30.5ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b01f8736cc0>
curves: ['Precision-Recall(B)

In [18]:
results = model.predict(
    source="/kaggle/input/datasets/tuyennguyen21pt/flight",
    conf=0.25,
    save=True
)


image 1/1 /kaggle/input/datasets/tuyennguyen21pt/flight/180_flight_406_5110_png.rf.05d8ec52d1f32e716f861258e8d02197.jpg: 384x640 32 boxs, 49.3ms
Speed: 2.7ms preprocess, 49.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /kaggle/working/runs/detect/predict


In [19]:
results = model.predict(
    "/kaggle/input/datasets/tuyennguyen21pt/flight"
)

count = len(results[0].boxes)

print("Number of boxes:", count)


image 1/1 /kaggle/input/datasets/tuyennguyen21pt/flight/180_flight_406_5110_png.rf.05d8ec52d1f32e716f861258e8d02197.jpg: 384x640 32 boxs, 23.9ms
Speed: 2.3ms preprocess, 23.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
Number of boxes: 32


In [ ]:
import shutil
from pathlib import Path
from ultralytics import YOLO

# Tìm best.pt của lần train gần nhất.
best = sorted(Path("/kaggle/working/runs/detect").glob("**/weights/best.pt"),
              key=lambda p: p.stat().st_mtime)[-1]
print("best.pt:", best)

# XÁC NHẬN class names — đây là điểm mấu chốt.
# Phải in ra {0: 'box'}. Nếu thấy {0: '-', 1: 'box'} nghĩa là data.yaml/nhãn còn sai,
# và service (lọc theo tên lớp) sẽ không đếm được gì.
check = YOLO(str(best))
print("names:", check.names)
assert check.names == {0: "box"}, f"Sai class names: {check.names} (mong đợi {{0: 'box'}})"

# Copy ra /kaggle/working để tải về và thay vào box-counter-service/model/best.pt
out = Path("/kaggle/working/best.pt")
shutil.copy(best, out)
print("Đã export:", out)
print("=> Tải file này thay cho box-counter-service/model/best.pt")
print("=> Vì model giờ có class 'box', đặt lại BOX_CLASS_NAME=box trong docker-compose.yml")